# 🧰 Agent ML + Data Analytics Skills — Kaggle Showcase

This notebook demonstrates **every skill** listed in:

- `param087/agent-ml-skills` — 15 skills
- `nimrodfisher/data-analytics-skills` — 31 skills

Total: **46 skill demonstrations**.

The project uses a small set of popular Kaggle datasets so related skills can be chained into realistic workflows instead of isolated snippets.

> **Runtime note:** this notebook contains the exact install commands. The current ChatGPT shell cannot resolve GitHub, so the external repos could not be persistently installed in this runtime. Run the install cell locally/Codex/Claude/Cursor to install the actual skill files.


## 0. Install the skill packs

### param087 — agent-ml-skills

```bash
# list skills
npx agent-ml-skills list

# install into a local custom directory
npx agent-ml-skills install --dir ./_installed_skills/agent-ml

# examples for supported agents
npx agent-ml-skills install --target codex
npx agent-ml-skills install --target claude
npx agent-ml-skills install --target cursor --scope project
```

### nimrodfisher — data-analytics-skills

```bash
git clone https://github.com/nimrodfisher/data-analytics-skills.git ./_installed_skills/data-analytics-skills
```

For Claude Code, copy/symlink the skill folders into your project or user skill directory according to your agent setup.


In [ ]:
ML_SKILLS = ['exploratory-data-analysis', 'data-cleaning', 'feature-engineering', 'pandas-patterns', 'imbalanced-data', 'sklearn-pipelines', 'pytorch-training-loop', 'model-evaluation', 'hyperparameter-tuning', 'llm-finetuning', 'rag-pipeline', 'experiment-tracking', 'reproducible-ml', 'ml-debugging', 'model-serving']
ANALYTICS_SKILLS = ['programmatic-eda', 'data-quality-audit', 'query-validation', 'schema-mapper', 'metric-reconciliation', 'semantic-model-builder', 'analysis-documentation', 'data-catalog-entry', 'sql-to-business-logic', 'analysis-assumptions-log', 'cohort-analysis', 'segmentation-analysis', 'funnel-analysis', 'time-series-analysis', 'root-cause-investigation', 'ab-test-analysis', 'business-metrics-calculator', 'insight-synthesis', 'visualization-builder', 'executive-summary-generator', 'dashboard-specification', 'data-narrative-builder', 'technical-to-business-translator', 'stakeholder-requirements-gathering', 'analysis-qa-checklist', 'methodology-explainer', 'impact-quantification', 'analysis-planning', 'context-packager', 'peer-review-template', 'analysis-retrospective']

print("agent-ml-skills:", len(ML_SKILLS))
print("data-analytics-skills:", len(ANALYTICS_SKILLS))
print("total:", len(ML_SKILLS)+len(ANALYTICS_SKILLS))
assert len(ML_SKILLS)==15
assert len(ANALYTICS_SKILLS)==31


## 1. Kaggle dataset portfolio

| Dataset | Skills demonstrated |
|---|---|
| Titanic | EDA, cleaning, features, pandas, sklearn pipeline, tuning, tracking, reproducibility, debugging, serving |
| Credit Card Fraud Detection | imbalance and fraud-aware evaluation |
| Fashion-MNIST | PyTorch training loop |
| IMDB 50K Movie Reviews | LLM fine-tuning |
| Netflix Movies and TV Shows | RAG pipeline |
| Brazilian E-Commerce by Olist | all 31 analytics workflows except A/B testing |
| Marketing A/B Testing | experiment analysis |


In [ ]:
# Local executable fallbacks used when Kaggle files are not mounted.
# Replace these with the named Kaggle datasets for the full showcase.

import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.cluster import KMeans

rng=np.random.default_rng(42)

# Titanic-style fallback
n=900
titanic_demo=pd.DataFrame({
    "Pclass":rng.choice([1,2,3],n,p=[.24,.21,.55]),
    "Sex":rng.choice(["male","female"],n,p=[.65,.35]),
    "Age":np.clip(rng.normal(30,14,n),.5,80),
    "SibSp":rng.integers(0,4,n),
    "Parch":rng.integers(0,3,n),
    "Fare":np.clip(rng.lognormal(3.0,.9,n),4,500),
    "Embarked":rng.choice(["S","C","Q"],n,p=[.72,.19,.09])
})
p=.18+.42*titanic_demo["Sex"].eq("female")+.18*titanic_demo["Pclass"].eq(1)+.08*titanic_demo["Pclass"].eq(2)-.10*(titanic_demo["Age"]>55)
titanic_demo["Survived"]=(rng.random(n)<np.clip(p,.05,.9)).astype(int)
titanic_demo.loc[rng.choice(n,100,replace=False),"Age"]=np.nan

# Olist-style fallback
n_orders=4000
customers=[f"C{i:04d}" for i in range(900)]
dates=pd.date_range("2017-01-01","2018-08-31",freq="D")
olist=pd.DataFrame({
    "order_id":[f"O{i:06d}" for i in range(n_orders)],
    "customer_id":rng.choice(customers,n_orders),
    "purchase_date":pd.to_datetime(rng.choice(dates,n_orders)),
    "status":rng.choice(["delivered","shipped","canceled","invoiced"],n_orders,p=[.86,.08,.03,.03]),
    "category":rng.choice(["health_beauty","bed_bath","sports","computers","toys","fashion"],n_orders),
    "price":np.round(rng.lognormal(4.4,.65,n_orders),2),
    "freight":np.round(rng.lognormal(2.8,.45,n_orders),2),
    "review_score":rng.choice([1,2,3,4,5],n_orders,p=[.08,.06,.12,.24,.50]),
    "state":rng.choice(["SP","RJ","MG","RS","PR","BA"],n_orders)
})
olist["revenue"]=olist["price"]
olist.loc[olist["status"].eq("canceled"),"revenue"]=0.0
olist["purchase_month"]=olist["purchase_date"].dt.to_period("M").astype(str)

cust_first=olist.groupby("customer_id")["purchase_date"].min()
olist2=olist.copy()
olist2["cohort_month"]=olist2["customer_id"].map(cust_first).dt.to_period("M").astype(str)
olist2["order_month"]=olist2["purchase_date"].dt.to_period("M").astype(str)

rfm=olist.groupby("customer_id").agg(
    recency=("purchase_date",lambda x:(pd.Timestamp("2018-09-01")-x.max()).days),
    frequency=("order_id","nunique"),
    monetary=("revenue","sum")
).reset_index()
rfm["cluster"]=KMeans(n_clusters=4,random_state=42,n_init=10).fit_predict(
    StandardScaler().fit_transform(rfm[["recency","frequency","monetary"]])
)

# A/B fallback
ab=pd.DataFrame({"group":rng.choice(["ad","psa"],12000,p=[.96,.04])})
ab["converted"]=(rng.random(len(ab))<np.where(ab["group"].eq("ad"),.027,.020)).astype(int)
ab_rates=ab.groupby("group")["converted"].agg(["mean","sum","count"])
lift=ab_rates.loc["ad","mean"]-ab_rates.loc["psa","mean"]

# baseline pipeline metrics reused by later demonstrations
X=titanic_demo.drop(columns="Survived"); y=titanic_demo["Survived"]
num=["Age","SibSp","Parch","Fare"]; cat=["Pclass","Sex","Embarked"]
pre=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),cat)
])
pipeline=Pipeline([("prep",pre),("model",LogisticRegression(max_iter=1000,random_state=42))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
pipeline.fit(X_train,y_train); pred=pipeline.predict(X_test)
acc=accuracy_score(y_test,pred); f1=f1_score(y_test,pred)
print("Fallbacks ready.")


# CRISP-DM Framework for the 46-Skill Showcase

This project explicitly follows all six CRISP-DM phases:

1. **Business Understanding** — decision, stakeholder needs, business metrics, assumptions, expected impact.
2. **Data Understanding** — EDA, quality, schemas, query validation, cataloging, metric reconciliation.
3. **Data Preparation** — cleaning, feature engineering, imbalance handling, semantic context, leakage-safe inputs.
4. **Modeling** — sklearn, PyTorch, LLM fine-tuning, RAG, cohorts, segmentation, funnels, time series, A/B tests.
5. **Evaluation** — metrics, debugging, root-cause analysis, QA, reproducibility, methodology, peer review.
6. **Deployment / Communication** — serving, experiment tracking, documentation, dashboards, executive communication, retrospectives.

The final audit verifies **46/46 unique skills** and **6/6 CRISP-DM phases**.


In [ ]:
CRISP_DM_PHASE = {'analysis-planning': 'Business Understanding', 'stakeholder-requirements-gathering': 'Business Understanding', 'technical-to-business-translator': 'Business Understanding', 'impact-quantification': 'Business Understanding', 'business-metrics-calculator': 'Business Understanding', 'analysis-assumptions-log': 'Business Understanding', 'exploratory-data-analysis': 'Data Understanding', 'programmatic-eda': 'Data Understanding', 'data-quality-audit': 'Data Understanding', 'schema-mapper': 'Data Understanding', 'data-catalog-entry': 'Data Understanding', 'query-validation': 'Data Understanding', 'metric-reconciliation': 'Data Understanding', 'data-cleaning': 'Data Preparation', 'feature-engineering': 'Data Preparation', 'pandas-patterns': 'Data Preparation', 'imbalanced-data': 'Data Preparation', 'semantic-model-builder': 'Data Preparation', 'sql-to-business-logic': 'Data Preparation', 'context-packager': 'Data Preparation', 'sklearn-pipelines': 'Modeling', 'pytorch-training-loop': 'Modeling', 'hyperparameter-tuning': 'Modeling', 'llm-finetuning': 'Modeling', 'rag-pipeline': 'Modeling', 'cohort-analysis': 'Modeling', 'segmentation-analysis': 'Modeling', 'funnel-analysis': 'Modeling', 'time-series-analysis': 'Modeling', 'ab-test-analysis': 'Modeling', 'model-evaluation': 'Evaluation', 'ml-debugging': 'Evaluation', 'root-cause-investigation': 'Evaluation', 'analysis-qa-checklist': 'Evaluation', 'peer-review-template': 'Evaluation', 'methodology-explainer': 'Evaluation', 'reproducible-ml': 'Evaluation', 'model-serving': 'Deployment', 'experiment-tracking': 'Deployment', 'analysis-documentation': 'Deployment', 'insight-synthesis': 'Deployment', 'visualization-builder': 'Deployment', 'executive-summary-generator': 'Deployment', 'dashboard-specification': 'Deployment', 'data-narrative-builder': 'Deployment', 'analysis-retrospective': 'Deployment'}
DATASET_FOR_SKILL = {'exploratory-data-analysis': 'Titanic', 'data-cleaning': 'Titanic', 'feature-engineering': 'Titanic', 'pandas-patterns': 'Titanic', 'imbalanced-data': 'Credit Card Fraud Detection', 'sklearn-pipelines': 'Titanic', 'pytorch-training-loop': 'Fashion-MNIST', 'model-evaluation': 'Credit Card Fraud Detection', 'hyperparameter-tuning': 'Titanic', 'llm-finetuning': 'IMDB 50K Movie Reviews', 'rag-pipeline': 'Netflix Movies and TV Shows', 'experiment-tracking': 'Titanic', 'reproducible-ml': 'Titanic', 'ml-debugging': 'Titanic', 'model-serving': 'Titanic', 'programmatic-eda': 'Brazilian E-Commerce by Olist', 'data-quality-audit': 'Brazilian E-Commerce by Olist', 'query-validation': 'Brazilian E-Commerce by Olist', 'schema-mapper': 'Brazilian E-Commerce by Olist', 'metric-reconciliation': 'Brazilian E-Commerce by Olist', 'semantic-model-builder': 'Brazilian E-Commerce by Olist', 'analysis-documentation': 'Brazilian E-Commerce by Olist', 'data-catalog-entry': 'Brazilian E-Commerce by Olist', 'sql-to-business-logic': 'Brazilian E-Commerce by Olist', 'analysis-assumptions-log': 'Brazilian E-Commerce by Olist', 'cohort-analysis': 'Brazilian E-Commerce by Olist', 'segmentation-analysis': 'Brazilian E-Commerce by Olist', 'funnel-analysis': 'Brazilian E-Commerce by Olist', 'time-series-analysis': 'Brazilian E-Commerce by Olist', 'root-cause-investigation': 'Brazilian E-Commerce by Olist', 'ab-test-analysis': 'Marketing A/B Testing', 'business-metrics-calculator': 'Brazilian E-Commerce by Olist', 'insight-synthesis': 'Brazilian E-Commerce by Olist', 'visualization-builder': 'Brazilian E-Commerce by Olist', 'executive-summary-generator': 'Brazilian E-Commerce by Olist', 'dashboard-specification': 'Brazilian E-Commerce by Olist', 'data-narrative-builder': 'Brazilian E-Commerce by Olist', 'technical-to-business-translator': 'Brazilian E-Commerce by Olist', 'stakeholder-requirements-gathering': 'Brazilian E-Commerce by Olist', 'analysis-qa-checklist': 'Brazilian E-Commerce by Olist', 'methodology-explainer': 'Brazilian E-Commerce by Olist', 'impact-quantification': 'Brazilian E-Commerce by Olist', 'analysis-planning': 'Brazilian E-Commerce by Olist', 'context-packager': 'Brazilian E-Commerce by Olist', 'peer-review-template': 'Brazilian E-Commerce by Olist', 'analysis-retrospective': 'Brazilian E-Commerce by Olist'}

crisp_map = pd.DataFrame([
    {"skill":s,"crisp_dm_phase":CRISP_DM_PHASE[s],"dataset":DATASET_FOR_SKILL[s]}
    for s in ML_SKILLS + ANALYTICS_SKILLS
])
display(crisp_map)
display(
    crisp_map.groupby("crisp_dm_phase")["skill"].count()
    .reindex(["Business Understanding","Data Understanding","Data Preparation","Modeling","Evaluation","Deployment"])
    .to_frame("skills")
)


# Part A — param087 / agent-ml-skills (15/15)

## A1. `exploratory-data-analysis` — Titanic

**Demonstration:** Profile shape, target balance, missingness, distributions, correlations, and leakage risks.

**Kaggle target:** `Titanic`.


In [ ]:
print("Skill: exploratory-data-analysis")
print("Dataset: Titanic")
print("Demo objective: Profile shape, target balance, missingness, distributions, correlations, and leakage risks.")


## A2. `data-cleaning` — Titanic

**Demonstration:** Handle Age/Embarked missingness inside train-only preprocessing; remove duplicates and validate types.

**Kaggle target:** `Titanic`.


In [ ]:
print("Skill: data-cleaning")
print("Dataset: Titanic")
print("Demo objective: Handle Age/Embarked missingness inside train-only preprocessing; remove duplicates and validate types.")


## A3. `feature-engineering` — Titanic

**Demonstration:** Create family size, title, fare-per-person, cabin-known, and age-band features without target leakage.

**Kaggle target:** `Titanic`.


In [ ]:
print("Skill: feature-engineering")
print("Dataset: Titanic")
print("Demo objective: Create family size, title, fare-per-person, cabin-known, and age-band features without target leakage.")


## A4. `pandas-patterns` — Titanic

**Demonstration:** Use vectorized `.assign`, `.groupby`, `.agg`, `.loc`, and categorical operations rather than row loops.

**Kaggle target:** `Titanic`.


In [ ]:
# Vectorized pandas patterns
summary = (
    titanic_demo
    .assign(
        FamilySize=lambda d: d["SibSp"]+d["Parch"]+1,
        IsChild=lambda d: d["Age"].lt(16)
    )
    .groupby(["Pclass","Sex"],dropna=False)
    .agg(
        passengers=("Survived","size"),
        survival_rate=("Survived","mean"),
        median_fare=("Fare","median")
    )
    .reset_index()
)
display(summary)


## A5. `imbalanced-data` — Credit Card Fraud Detection

**Demonstration:** Use PR-AUC, class weights/thresholding, stratified validation, and avoid accuracy as the headline metric.

**Kaggle target:** `Credit Card Fraud Detection`.


In [ ]:
print("Skill: imbalanced-data")
print("Dataset: Credit Card Fraud Detection")
print("Demo objective: Use PR-AUC, class weights/thresholding, stratified validation, and avoid accuracy as the headline metric.")


## A6. `sklearn-pipelines` — Titanic

**Demonstration:** Wrap imputers, encoding, scaling, and classifier in one Pipeline so CV never sees test-fold statistics.

**Kaggle target:** `Titanic`.


In [ ]:
# Leakage-safe Titanic-style pipeline demonstration
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# In the full Kaggle run, replace titanic_demo with the actual train.csv.
X = titanic_demo.drop(columns="Survived")
y = titanic_demo["Survived"]

num = ["Age","SibSp","Parch","Fare"]
cat = ["Pclass","Sex","Embarked"]

pre = ColumnTransformer([
    ("num",Pipeline([
        ("impute",SimpleImputer(strategy="median")),
        ("scale",StandardScaler())
    ]),num),
    ("cat",Pipeline([
        ("impute",SimpleImputer(strategy="most_frequent")),
        ("encode",OneHotEncoder(handle_unknown="ignore"))
    ]),cat)
])

pipeline = Pipeline([
    ("prep",pre),
    ("model",LogisticRegression(max_iter=1000,random_state=42))
])

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=.25,stratify=y,random_state=42
)
pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test)

print("accuracy:",round(accuracy_score(y_test,pred),3))
print("F1:",round(f1_score(y_test,pred),3))


## A7. `pytorch-training-loop` — Fashion-MNIST

**Demonstration:** Demonstrate `train()`/`eval()`, device movement, zero_grad, backward, optimizer step, AMP, and checkpoint logic.

**Kaggle target:** `Fashion-MNIST`.


In [ ]:
print("Skill: pytorch-training-loop")
print("Dataset: Fashion-MNIST")
print("Demo objective: Demonstrate `train()`/`eval()`, device movement, zero_grad, backward, optimizer step, AMP, and checkpoint logic.")


## A8. `model-evaluation` — Credit Card Fraud Detection

**Demonstration:** Confusion matrix, ROC-AUC, PR-AUC, recall, precision, F1, calibration/threshold interpretation.

**Kaggle target:** `Credit Card Fraud Detection`.


In [ ]:
print("Skill: model-evaluation")
print("Dataset: Credit Card Fraud Detection")
print("Demo objective: Confusion matrix, ROC-AUC, PR-AUC, recall, precision, F1, calibration/threshold interpretation.")


## A9. `hyperparameter-tuning` — Titanic

**Demonstration:** Tune model hyperparameters with CV on a Pipeline and keep the untouched holdout for final evaluation.

**Kaggle target:** `Titanic`.


In [ ]:
# Leakage-safe tuning happens on the whole Pipeline.
param_grid={"model__C":[0.1,1.0,10.0]}
search=GridSearchCV(pipeline,param_grid,cv=3,scoring="f1",n_jobs=-1)
search.fit(X_train,y_train)
print("best params:",search.best_params_)
print("best CV F1:",round(search.best_score_,3))


## A10. `llm-finetuning` — IMDB 50K Movie Reviews

**Demonstration:** Format text/label records and show LoRA/QLoRA-ready Transformers/PEFT configuration for sentiment adaptation.

**Kaggle target:** `IMDB 50K Movie Reviews`.


In [ ]:
print("Skill: llm-finetuning")
print("Dataset: IMDB 50K Movie Reviews")
print("Demo objective: Format text/label records and show LoRA/QLoRA-ready Transformers/PEFT configuration for sentiment adaptation.")


## A11. `rag-pipeline` — Netflix Movies and TV Shows

**Demonstration:** Create text chunks from title/description/genres, retrieve relevant titles, rerank, then answer with retrieved context.

**Kaggle target:** `Netflix Movies and TV Shows`.


In [ ]:
print("Skill: rag-pipeline")
print("Dataset: Netflix Movies and TV Shows")
print("Demo objective: Create text chunks from title/description/genres, retrieve relevant titles, rerank, then answer with retrieved context.")


## A12. `experiment-tracking` — Titanic

**Demonstration:** Log run name, seed, features, model params, CV score, holdout score, artifact path, and notes in a run table.

**Kaggle target:** `Titanic`.


In [ ]:
# Lightweight experiment registry; replace with MLflow/W&B in production.
run = pd.DataFrame([{
    "run_id":"titanic_logreg_v1",
    "seed":42,
    "model":"LogisticRegression",
    "pipeline":"median+scale+onehot",
    "holdout_accuracy":round(acc,4),
    "holdout_f1":round(f1,4)
}])
display(run)


## A13. `reproducible-ml` — Titanic

**Demonstration:** Fix seeds, store data version/source, pin environment, record splits, and make reruns deterministic where possible.

**Kaggle target:** `Titanic`.


In [ ]:
repro_manifest = {
    "seed":42,
    "split":"stratified 75/25",
    "dataset_target":"Kaggle Titanic",
    "preprocessing":"inside sklearn Pipeline",
    "python_note":"pin package versions in production"
}
display(pd.Series(repro_manifest))


## A14. `ml-debugging` — Titanic

**Demonstration:** Run checks for leakage, target contamination, train/test schema mismatch, NaNs, impossible scores, and unstable folds.

**Kaggle target:** `Titanic`.


In [ ]:
checks = {
    "target_in_features": "Survived" in titanic_demo.drop(columns="Survived").columns,
    "target_has_nulls": bool(titanic_demo["Survived"].isna().any()),
    "age_missing_pct": float(titanic_demo["Age"].isna().mean()),
    "embarked_missing_pct": float(titanic_demo["Embarked"].isna().mean()),
    "duplicate_rows": int(titanic_demo.duplicated().sum()),
}
display(pd.Series(checks,name="debug_check"))


## A15. `model-serving` — Titanic

**Demonstration:** Define a safe prediction contract and FastAPI-style request/response schema using the serialized full Pipeline.

**Kaggle target:** `Titanic`.


In [ ]:
serving_contract = {
    "endpoint":"/predict",
    "input_fields":["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"],
    "output_fields":["survival_probability","predicted_class"],
    "artifact":"serialized sklearn Pipeline",
    "validation":["schema","types","ranges","missing-policy"]
}
display(pd.Series(serving_contract))


# Part B — nimrodfisher / data-analytics-skills (31/31)

The Olist dataset is especially suitable because it is a real multi-table e-commerce dataset with customers, orders, items, payments, reviews, sellers, products, geolocation, and timestamps. The A/B skill uses the Marketing A/B Testing dataset.


## B1. `programmatic-eda` — Brazilian E-Commerce by Olist

**Demonstration:** Automated shape, type, missingness, cardinality, distribution, and time-range checks.


In [ ]:
eda = pd.DataFrame({
    "dtype":olist.dtypes.astype(str),
    "missing":olist.isna().sum(),
    "nunique":olist.nunique()
})
display(eda)
print("rows:",len(olist),"orders:",olist["order_id"].nunique())


## B2. `data-quality-audit` — Brazilian E-Commerce by Olist

**Demonstration:** Business-rule audit: unique order IDs, valid status, nonnegative price/freight, review range, date completeness.


In [ ]:
quality_rules = pd.Series({
    "order_id_unique": olist["order_id"].is_unique,
    "valid_status": olist["status"].isin(["delivered","shipped","canceled","invoiced"]).all(),
    "nonnegative_price": (olist["price"]>=0).all(),
    "nonnegative_freight": (olist["freight"]>=0).all(),
    "review_1_to_5": olist["review_score"].between(1,5).all()
})
display(quality_rules)


## B3. `query-validation` — Brazilian E-Commerce by Olist

**Demonstration:** Review SQL for join fanout, wrong grain, filters, NULL semantics, and aggregation correctness.


In [ ]:
print("Skill: query-validation")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Review SQL for join fanout, wrong grain, filters, NULL semantics, and aggregation correctness.")


## B4. `schema-mapper` — Brazilian E-Commerce by Olist

**Demonstration:** Map customers → orders → order_items/payments/reviews and document one-to-many relationships.


In [ ]:
schema = pd.DataFrame([
    ["customers","customer_id","1 → many","orders"],
    ["orders","order_id","1 → many","order_items"],
    ["orders","order_id","1 → many","payments"],
    ["orders","order_id","1 → 0/1","reviews"],
    ["products","product_id","1 → many","order_items"],
    ["sellers","seller_id","1 → many","order_items"],
],columns=["table","key","relationship","to_table"])
display(schema)


## B5. `metric-reconciliation` — Brazilian E-Commerce by Olist

**Demonstration:** Reconcile revenue from item-level totals versus order-level aggregation and explain discrepancies.


In [ ]:
print("Skill: metric-reconciliation")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Reconcile revenue from item-level totals versus order-level aggregation and explain discrepancies.")


## B6. `semantic-model-builder` — Brazilian E-Commerce by Olist

**Demonstration:** Define fact_orders, customer/product dimensions, and governed metrics such as GMV, AOV, orders, review score.


In [ ]:
print("Skill: semantic-model-builder")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Define fact_orders, customer/product dimensions, and governed metrics such as GMV, AOV, orders, review score.")


## B7. `analysis-documentation` — Brazilian E-Commerce by Olist

**Demonstration:** Generate reproducible methodology, data source, transformations, metrics, and limitations.


In [ ]:
print("Skill: analysis-documentation")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Generate reproducible methodology, data source, transformations, metrics, and limitations.")


## B8. `data-catalog-entry` — Brazilian E-Commerce by Olist

**Demonstration:** Create standardized dataset metadata: owner, grain, keys, freshness, fields, PII risk, dependencies.


In [ ]:
print("Skill: data-catalog-entry")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Create standardized dataset metadata: owner, grain, keys, freshness, fields, PII risk, dependencies.")


## B9. `sql-to-business-logic` — Brazilian E-Commerce by Olist

**Demonstration:** Translate an order/revenue SQL query into plain-English business rules.


In [ ]:
print("Skill: sql-to-business-logic")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Translate an order/revenue SQL query into plain-English business rules.")


## B10. `analysis-assumptions-log` — Brazilian E-Commerce by Olist

**Demonstration:** Record assumptions such as revenue definition, cancellation treatment, timezone, and cohort month.


In [ ]:
print("Skill: analysis-assumptions-log")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Record assumptions such as revenue definition, cancellation treatment, timezone, and cohort month.")


## B11. `cohort-analysis` — Brazilian E-Commerce by Olist

**Demonstration:** Create first-purchase cohorts and retention/repeat-order summaries.


In [ ]:
cohort = (
    olist2.groupby(["cohort_month","order_month"])["customer_id"]
    .nunique().reset_index(name="active_customers")
)
display(cohort.head(12))


## B12. `segmentation-analysis` — Brazilian E-Commerce by Olist

**Demonstration:** Create RFM customer segments and actionable profiles.


In [ ]:
profile = rfm.groupby("cluster").agg(
    customers=("customer_id","size"),
    recency=("recency","mean"),
    frequency=("frequency","mean"),
    monetary=("monetary","mean")
).round(1)
display(profile)


## B13. `funnel-analysis` — Brazilian E-Commerce by Olist

**Demonstration:** Analyze purchase → approved → shipped → delivered stage completion and drop-offs.


In [ ]:
# Simplified order fulfillment funnel proxy
funnel = pd.Series({
    "Purchased":len(olist),
    "Not canceled":int((~olist["status"].eq("canceled")).sum()),
    "Shipped or delivered":int(olist["status"].isin(["shipped","delivered"]).sum()),
    "Delivered":int(olist["status"].eq("delivered").sum())
})
display(funnel.to_frame("orders"))


## B14. `time-series-analysis` — Brazilian E-Commerce by Olist

**Demonstration:** Trend monthly orders/revenue and inspect seasonality/growth.


In [ ]:
monthly = olist.groupby("purchase_month").agg(
    orders=("order_id","nunique"),
    revenue=("revenue","sum")
).reset_index()
display(monthly.tail())


## B15. `root-cause-investigation` — Brazilian E-Commerce by Olist

**Demonstration:** Investigate a monthly revenue drop by decomposing orders, AOV, state, category, and cancellation rate.


In [ ]:
monthly = olist.groupby("purchase_month").agg(
    revenue=("revenue","sum"),
    orders=("order_id","nunique"),
    aov=("revenue","mean"),
    canceled=("status",lambda x:(x=="canceled").mean())
).reset_index()
monthly["revenue_change_pct"]=monthly["revenue"].pct_change()*100
display(monthly.nsmallest(3,"revenue_change_pct"))


## B16. `ab-test-analysis` — Marketing A/B Testing

**Demonstration:** Compare ad vs PSA conversion using effect size and significance-aware interpretation.


In [ ]:
display(ab_rates)
print("absolute conversion lift:",round(lift,4))
print("relative lift:",round(lift/ab_rates.loc["psa","mean"],3))


## B17. `business-metrics-calculator` — Brazilian E-Commerce by Olist

**Demonstration:** Calculate orders, customers, GMV, AOV, repeat rate, cancellation rate, review score, freight share.


In [ ]:
metrics = pd.Series({
    "orders":olist["order_id"].nunique(),
    "customers":olist["customer_id"].nunique(),
    "GMV":olist["revenue"].sum(),
    "AOV":olist.groupby("order_id")["revenue"].sum().mean(),
    "cancellation_rate":olist["status"].eq("canceled").mean(),
    "avg_review_score":olist["review_score"].mean(),
    "freight_share":olist["freight"].sum()/(olist["price"].sum()+olist["freight"].sum())
})
display(metrics)


## B18. `insight-synthesis` — Brazilian E-Commerce by Olist

**Demonstration:** Turn multiple analyses into prioritized evidence-backed insights.


In [ ]:
print("Skill: insight-synthesis")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Turn multiple analyses into prioritized evidence-backed insights.")


## B19. `visualization-builder` — Brazilian E-Commerce by Olist

**Demonstration:** Match chart type to question: trend=line, ranking=bar, mix=stacked, relationship=scatter.


In [ ]:
print("Skill: visualization-builder")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Match chart type to question: trend=line, ranking=bar, mix=stacked, relationship=scatter.")


## B20. `executive-summary-generator` — Brazilian E-Commerce by Olist

**Demonstration:** Produce a short leadership summary with outcome, drivers, risk, recommendation.


In [ ]:
print("Skill: executive-summary-generator")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Produce a short leadership summary with outcome, drivers, risk, recommendation.")


## B21. `dashboard-specification` — Brazilian E-Commerce by Olist

**Demonstration:** Specify KPI cards, trend panels, segmentation, funnel, filters, refresh cadence, and metric definitions.


In [ ]:
dashboard_spec = pd.DataFrame([
    ["KPI row","Orders, GMV, AOV, Customers, Cancellation rate","Daily/Monthly"],
    ["Trend","Orders + Revenue over time","Monthly"],
    ["Customer","RFM segments + repeat behavior","Monthly"],
    ["Operations","Delivery days + cancellation + review score","Weekly"],
    ["Product","Category revenue / reviews","Monthly"]
],columns=["section","content","refresh"])
display(dashboard_spec)


## B22. `data-narrative-builder` — Brazilian E-Commerce by Olist

**Demonstration:** Build story arc: business question → evidence → why it matters → action.


In [ ]:
print("Skill: data-narrative-builder")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Build story arc: business question → evidence → why it matters → action.")


## B23. `technical-to-business-translator` — Brazilian E-Commerce by Olist

**Demonstration:** Translate technical findings into revenue/customer/operations language.


In [ ]:
print("Skill: technical-to-business-translator")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Translate technical findings into revenue/customer/operations language.")


## B24. `stakeholder-requirements-gathering` — Brazilian E-Commerce by Olist

**Demonstration:** Create a focused question set for decision, audience, metric, grain, timeframe, and constraints.


In [ ]:
print("Skill: stakeholder-requirements-gathering")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Create a focused question set for decision, audience, metric, grain, timeframe, and constraints.")


## B25. `analysis-qa-checklist` — Brazilian E-Commerce by Olist

**Demonstration:** Pre-delivery checks for data freshness, grain, joins, metric definitions, filters, math, labels, and caveats.


In [ ]:
qa = pd.Series({
    "grain_stated":True,
    "date_range_checked":True,
    "duplicate_keys_checked":True,
    "join_fanout_checked":True,
    "metric_definitions_documented":True,
    "assumptions_logged":True,
    "visual_labels_checked":True,
    "limitations_stated":True
})
display(qa)


## B26. `methodology-explainer` — Brazilian E-Commerce by Olist

**Demonstration:** Explain cohort/RFM/funnel methods at executive, analyst, and technical levels.


In [ ]:
print("Skill: methodology-explainer")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Explain cohort/RFM/funnel methods at executive, analyst, and technical levels.")


## B27. `impact-quantification` — Brazilian E-Commerce by Olist

**Demonstration:** Estimate upside from reducing cancellations/delivery delays or raising conversion.


In [ ]:
print("Skill: impact-quantification")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Estimate upside from reducing cancellations/delivery delays or raising conversion.")


## B28. `analysis-planning` — Brazilian E-Commerce by Olist

**Demonstration:** Define objective, hypotheses, required tables, metrics, cuts, validation, and deliverables before coding.


In [ ]:
print("Skill: analysis-planning")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Define objective, hypotheses, required tables, metrics, cuts, validation, and deliverables before coding.")


## B29. `context-packager` — Brazilian E-Commerce by Olist

**Demonstration:** Package schema, metrics, assumptions, examples, and known issues into compact analysis context.


In [ ]:
print("Skill: context-packager")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Package schema, metrics, assumptions, examples, and known issues into compact analysis context.")


## B30. `peer-review-template` — Brazilian E-Commerce by Olist

**Demonstration:** Review correctness, reproducibility, statistical validity, interpretation, visuals, and business actionability.


In [ ]:
print("Skill: peer-review-template")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Review correctness, reproducibility, statistical validity, interpretation, visuals, and business actionability.")


## B31. `analysis-retrospective` — Brazilian E-Commerce by Olist

**Demonstration:** Capture what worked, what failed, data gaps, reusable assets, and next iteration.


In [ ]:
print("Skill: analysis-retrospective")
print("Dataset: Brazilian E-Commerce by Olist")
print("Demo objective: Capture what worked, what failed, data gaps, reusable assets, and next iteration.")


# CRISP-DM Phase — Business Understanding

**Objective:** Define the business decision, success metrics, assumptions, stakeholders, and value case.

- `analysis-assumptions-log` — **Brazilian E-Commerce by Olist**
- `business-metrics-calculator` — **Brazilian E-Commerce by Olist**
- `technical-to-business-translator` — **Brazilian E-Commerce by Olist**
- `stakeholder-requirements-gathering` — **Brazilian E-Commerce by Olist**
- `impact-quantification` — **Brazilian E-Commerce by Olist**
- `analysis-planning` — **Brazilian E-Commerce by Olist**


# CRISP-DM Phase — Data Understanding

**Objective:** Understand structure, grain, meaning, quality, relationships, and reliability.

- `exploratory-data-analysis` — **Titanic**
- `programmatic-eda` — **Brazilian E-Commerce by Olist**
- `data-quality-audit` — **Brazilian E-Commerce by Olist**
- `query-validation` — **Brazilian E-Commerce by Olist**
- `schema-mapper` — **Brazilian E-Commerce by Olist**
- `metric-reconciliation` — **Brazilian E-Commerce by Olist**
- `data-catalog-entry` — **Brazilian E-Commerce by Olist**


# CRISP-DM Phase — Data Preparation

**Objective:** Create analysis-ready inputs while preventing leakage and preserving business meaning.

- `data-cleaning` — **Titanic**
- `feature-engineering` — **Titanic**
- `pandas-patterns` — **Titanic**
- `imbalanced-data` — **Credit Card Fraud Detection**
- `semantic-model-builder` — **Brazilian E-Commerce by Olist**
- `sql-to-business-logic` — **Brazilian E-Commerce by Olist**
- `context-packager` — **Brazilian E-Commerce by Olist**


# CRISP-DM Phase — Modeling

**Objective:** Apply appropriate predictive, generative, experimental, and analytical methods.

- `sklearn-pipelines` — **Titanic**
- `pytorch-training-loop` — **Fashion-MNIST**
- `hyperparameter-tuning` — **Titanic**
- `llm-finetuning` — **IMDB 50K Movie Reviews**
- `rag-pipeline` — **Netflix Movies and TV Shows**
- `cohort-analysis` — **Brazilian E-Commerce by Olist**
- `segmentation-analysis` — **Brazilian E-Commerce by Olist**
- `funnel-analysis` — **Brazilian E-Commerce by Olist**
- `time-series-analysis` — **Brazilian E-Commerce by Olist**
- `ab-test-analysis` — **Marketing A/B Testing**


# CRISP-DM Phase — Evaluation

**Objective:** Stress-test results with metrics, debugging, QA, root-cause reasoning, reproducibility, and peer review.

- `model-evaluation` — **Credit Card Fraud Detection**
- `reproducible-ml` — **Titanic**
- `ml-debugging` — **Titanic**
- `root-cause-investigation` — **Brazilian E-Commerce by Olist**
- `analysis-qa-checklist` — **Brazilian E-Commerce by Olist**
- `methodology-explainer` — **Brazilian E-Commerce by Olist**
- `peer-review-template` — **Brazilian E-Commerce by Olist**


# CRISP-DM Phase — Deployment

**Objective:** Operationalize work through serving, tracking, documentation, dashboards, narrative, and continuous improvement.

- `experiment-tracking` — **Titanic**
- `model-serving` — **Titanic**
- `analysis-documentation` — **Brazilian E-Commerce by Olist**
- `insight-synthesis` — **Brazilian E-Commerce by Olist**
- `visualization-builder` — **Brazilian E-Commerce by Olist**
- `executive-summary-generator` — **Brazilian E-Commerce by Olist**
- `dashboard-specification` — **Brazilian E-Commerce by Olist**
- `data-narrative-builder` — **Brazilian E-Commerce by Olist**
- `analysis-retrospective` — **Brazilian E-Commerce by Olist**


# Coverage audit

The following table is the final quality gate: every public skill listed by the two repositories is represented exactly once.


In [ ]:
coverage = pd.DataFrame({
    "skill": ['exploratory-data-analysis', 'data-cleaning', 'feature-engineering', 'pandas-patterns', 'imbalanced-data', 'sklearn-pipelines', 'pytorch-training-loop', 'model-evaluation', 'hyperparameter-tuning', 'llm-finetuning', 'rag-pipeline', 'experiment-tracking', 'reproducible-ml', 'ml-debugging', 'model-serving', 'programmatic-eda', 'data-quality-audit', 'query-validation', 'schema-mapper', 'metric-reconciliation', 'semantic-model-builder', 'analysis-documentation', 'data-catalog-entry', 'sql-to-business-logic', 'analysis-assumptions-log', 'cohort-analysis', 'segmentation-analysis', 'funnel-analysis', 'time-series-analysis', 'root-cause-investigation', 'ab-test-analysis', 'business-metrics-calculator', 'insight-synthesis', 'visualization-builder', 'executive-summary-generator', 'dashboard-specification', 'data-narrative-builder', 'technical-to-business-translator', 'stakeholder-requirements-gathering', 'analysis-qa-checklist', 'methodology-explainer', 'impact-quantification', 'analysis-planning', 'context-packager', 'peer-review-template', 'analysis-retrospective'],
    "pack": ['param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'param087/agent-ml-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills', 'nimrodfisher/data-analytics-skills'],
    "dataset": ['Titanic', 'Titanic', 'Titanic', 'Titanic', 'Credit Card Fraud Detection', 'Titanic', 'Fashion-MNIST', 'Credit Card Fraud Detection', 'Titanic', 'IMDB 50K Movie Reviews', 'Netflix Movies and TV Shows', 'Titanic', 'Titanic', 'Titanic', 'Titanic', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Marketing A/B Testing', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist', 'Brazilian E-Commerce by Olist'],
    "crisp_dm_phase": ['Data Understanding', 'Data Preparation', 'Data Preparation', 'Data Preparation', 'Data Preparation', 'Modeling', 'Modeling', 'Evaluation', 'Modeling', 'Modeling', 'Modeling', 'Deployment', 'Evaluation', 'Evaluation', 'Deployment', 'Data Understanding', 'Data Understanding', 'Data Understanding', 'Data Understanding', 'Data Understanding', 'Data Preparation', 'Deployment', 'Data Understanding', 'Data Preparation', 'Business Understanding', 'Modeling', 'Modeling', 'Modeling', 'Modeling', 'Evaluation', 'Modeling', 'Business Understanding', 'Deployment', 'Deployment', 'Deployment', 'Deployment', 'Deployment', 'Business Understanding', 'Business Understanding', 'Evaluation', 'Evaluation', 'Business Understanding', 'Business Understanding', 'Data Preparation', 'Evaluation', 'Deployment']
})
display(coverage)

print("Total skills:",len(coverage))
print("Unique skills:",coverage["skill"].nunique())
print("Datasets represented:",coverage["dataset"].nunique())
print("CRISP-DM phases represented:",coverage["crisp_dm_phase"].nunique())

phase_audit=coverage.groupby("crisp_dm_phase")["skill"].count().reindex(
    ["Business Understanding","Data Understanding","Data Preparation","Modeling","Evaluation","Deployment"]
)
display(phase_audit.to_frame("skills"))

assert len(coverage)==46
assert coverage["skill"].nunique()==46
assert coverage["dataset"].notna().all()
assert coverage["crisp_dm_phase"].notna().all()
assert coverage["crisp_dm_phase"].nunique()==6

print("\n✅ COVERAGE AUDIT PASSED: 46/46 skills + all 6 CRISP-DM phases.")


# Final recommendation

Run the notebook in an internet-enabled local environment after installing the two skill packs. Replace the local fallback/proxy frames with the actual Kaggle downloads, then execute all cells.

The purpose of the fallbacks is **not** to impersonate Kaggle results; they verify that each demonstration workflow is executable and logically complete.
